## **Imports**

In [1]:
import math
import numpy as np
import torch
import os
from os.path import join as pjoin
import random
import matplotlib
import pandas as pd
import csv

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, FFMpegFileWriter
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import mpl_toolkits.mplot3d.axes3d as p3

## **Visualize Function**

In [2]:
kinematic_chain = [[0, 2, 5, 8, 11], 
                       [0, 1, 4, 7, 10], 
                       [0, 3, 6, 9, 12, 15], 
                       [9, 14, 17, 19, 21], 
                       [9, 13, 16, 18, 20]]

def plot_3d_motion(save_path, kinematic_tree, mp_joints, title, figsize=(10, 10), fps=120, radius=4):

    title_sp = title.split(' ')
    if len(title_sp) > 20:
        title = '\n'.join([' '.join(title_sp[:10]), ' '.join(title_sp[10:20]), ' '.join(title_sp[20:])])
    elif len(title_sp) > 10:
        title = '\n'.join([' '.join(title_sp[:10]), ' '.join(title_sp[10:])])

    def init():
        ax.set_xlim3d([-radius / 4, radius / 4])
        ax.set_ylim3d([0, radius / 2])
        ax.set_zlim3d([0, radius / 2])
        fig.suptitle(title, fontsize=20)
        ax.grid(b=False)

    def plot_xzPlane(minx, maxx, miny, minz, maxz):
        ## Plot a plane XZ
        verts = [
            [minx, miny, minz],
            [minx, miny, maxz],
            [maxx, miny, maxz],
            [maxx, miny, minz]
        ]
        xz_plane = Poly3DCollection([verts])
        xz_plane.set_facecolor((0.5, 0.5, 0.5, 0.5))
        ax.add_collection3d(xz_plane)

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    init()

    mp_data = []
    frame_number = min([data.shape[0] for data in mp_joints])

    colors = ['red', 'green', 'black', 'red', 'blue',
              'darkblue', 'darkblue', 'darkblue', 'darkblue', 'darkblue',
              'darkred', 'darkred', 'darkred', 'darkred', 'darkred']

    mp_offset = list(range(-len(mp_joints)//2, len(mp_joints)//2, 1))
    mp_colors = [[colors[i]] * 15 for i in range(len(mp_offset))]

    for i,joints in enumerate(mp_joints):

        # (seq_len, joints_num, 3)
        data = joints.copy().reshape(len(joints), -1, 3)

        MINS = data.min(axis=0).min(axis=0)
        MAXS = data.max(axis=0).max(axis=0)

        height_offset = MINS[1]
        data[:, :, 1] -= height_offset

        mp_data.append({"joints":data,
                        "MINS":MINS,
                        "MAXS":MAXS,})

    def update(index):
        for line in ax.lines:
            line.remove()
        for collection in ax.collections:
            collection.remove()
        ax.view_init(elev=120, azim=-90)
        # plot_xzPlane(-3, 3, 0, -3, 3)
        for pid,data in enumerate(mp_data):
            for i, (chain, color) in enumerate(zip(kinematic_tree, mp_colors[pid])):
                if i < 5:
                    linewidth = 4.0
                else:
                    linewidth = 2.0
                ax.plot3D(data["joints"][index, chain, 0], data["joints"][index, chain, 1], data["joints"][index, chain, 2], linewidth=linewidth,
                          color=color)

        ax.set_axis_off()
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        
    ani = FuncAnimation(fig, update, frames=frame_number, interval=1000 / fps, repeat=False)
    ani.save(save_path, fps=fps)
    plt.close()
    
    print('done ! animation saved to {}'.format(save_path))

In [3]:
## data directory
data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'
motion_dir = pjoin(data_root, 'motions', 'motions')
text_dir = pjoin(data_root, 'texts', 'texts')

## pick a random file
fname = random.choice(os.listdir(motion_dir))

## load motion & text
motion = np.load(pjoin(motion_dir, fname))
print('motion shape:', motion.shape)
with open(pjoin(text_dir, fname.replace('.npy', '.txt'))) as fr:
    texts = fr.read().strip().split('\n')
    
print('Number of text descriptions:', len(texts))
## pick a random text for visualization
title = random.choice(texts)
print('title:', title)

motion shape: (31, 384)
Number of text descriptions: 3
title: they stand side by side, clenching their right and left hands, respectively, pointing towards the same direction.


In [4]:
## extract positions for both persons
pos_person_1 = motion[:, :22*3]
pos_person_2 = motion[:, 22*3+21*6:22*3+21*6+22*3]

## save animation
plot_3d_motion('./animation.gif', kinematic_chain, [pos_person_1, pos_person_2], title, radius=4.0, fps=30)

MovieWriter ffmpeg unavailable; using Pillow instead.


done ! animation saved to ./animation.gif


## **Evaluation Function**

In [5]:
def eval_recall(gt_df, pred_df, verbose=False):

    ks = list(range(1, 11))  # Recall@1 ... Recall@9

    rank_cols = list(pred_df.columns)

    ## convert to numpy
    gt = gt_df["candidate"].values
    preds = pred_df[rank_cols].values

    assert len(gt) == len(preds), "size mismatch"
    
    ## compute Recall@K
    recalls = {k: 0 for k in ks}
    n = len(gt)

    for i in range(n):
        for k in ks:
            if gt[i] in preds[i, :k]:
                recalls[k] += 1

    for k in ks:
        recalls[k] /= n

    ## weighted aggregation
    weights = {k: 1.0 / k for k in ks} # higher weight for smaller k
    weight_sum = sum(weights.values())

    final_score = sum(
        weights[k] * recalls[k] for k in ks
    ) / weight_sum
    
    if verbose:
        for k, val in recalls.items():
            print('recall@{} = {}'.format(k, round(val, 3)))

    return final_score

## **Generate Validation Batches**

In [6]:
def generate_val_batches(data_root, save_dir, batch_size, num_batches):
    
    os.makedirs(save_dir, exist_ok=True)

    with open(os.path.join(data_root, 'train.txt')) as fd:
        train_fnames = fd.read().strip().split('\n') 

    random_fnames = random.sample(train_fnames, k=batch_size * num_batches)
    random_batches = np.array(random_fnames).reshape(num_batches, batch_size)

    gt = [['query_id', 'candidate']]
    for file_idx, batch in enumerate(random_batches, start=1):
        os.makedirs(pjoin(save_dir, str(file_idx)), exist_ok=True)

        rdm_idx = random.randint(0, len(batch)-1)
        random_text_fname = batch[rdm_idx]

        with open(os.path.join(data_root, 'texts', 'texts', random_text_fname+'.txt')) as fd:
            file_texts = fd.read().strip().split('\n')
            random_text = random.choice(file_texts)

        with open(os.path.join(save_dir, str(file_idx), 'text.txt'), 'w') as fw:
            fw.write(random_text)

        for motion_idx, fname in enumerate(batch, start=1):
            if fname == random_text_fname:
                gt.append([file_idx, motion_idx])

            motion = np.load(os.path.join(data_root, 'motions', 'motions', fname+'.npy'))

            np.save(pjoin(save_dir, str(file_idx), 'motion_{}.npy'.format(motion_idx)), motion)


    with open(os.path.join(save_dir, 'gt.csv'), 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(gt)
        
    print('Done ! val batches saved at {} !'.format(save_dir))

In [7]:
## data dir
data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'

## dir to save batches to
save_dir = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val/'
    
## number of val batches & number of element per batch
batch_size = 32
num_batches = 30

## generate val batches
generate_val_batches(data_root, save_dir, batch_size, num_batches)

Done ! val batches saved at /home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val/ !


### **Evaluate random submission**

In [8]:
## generate random submission (30 queries × 10 candidates)
random_sub = np.array([
    np.random.choice(range(1, 33), size=10, replace=False)
    for _ in range(30)
])

## create DataFrame
random_sub_df = pd.DataFrame(
    random_sub,
    index=np.arange(1, 31),
    columns=[f'candidate_{i}' for i in range(1, 11)]
)

random_sub_df.index.name = 'query_id'
random_sub_df.head()

,candidate_1,candidate_2,candidate_3,candidate_4,candidate_5,candidate_6,candidate_7,candidate_8,candidate_9,candidate_10
query_id,,,,,,,,,,
1,16,26,27,7,12,3,8,23,15,13
2,19,28,6,5,3,26,14,22,32,2
3,13,6,12,18,3,28,21,27,24,22
4,17,13,12,24,23,21,32,29,15,26
5,16,13,7,20,23,12,31,5,32,9


In [9]:
## load gt
gt_df = pd.read_csv(pjoin(save_dir, 'gt.csv'))
gt_df.head()

,query_id,candidate
0,1,22
1,2,10
2,3,3
3,4,32
4,5,4


In [10]:
## evaluate random results
racall = eval_recall(gt_df, random_sub_df, verbose=True)
print('Recall score:', racall)

recall@1 = 0.0
recall@2 = 0.033
recall@3 = 0.067
recall@4 = 0.1
recall@5 = 0.133
recall@6 = 0.133
recall@7 = 0.267
recall@8 = 0.3
recall@9 = 0.3
recall@10 = 0.333
Recall score: 0.08707492209727678


In [11]:
# def generate_val_batches_with_split(data_root, save_dir, batch_size, num_batches, val_ratio=0.1, seed=42):
#     random.seed(seed)
#     np.random.seed(seed)

#     os.makedirs(save_dir, exist_ok=True)

#     # --- 1. Charger toutes les motions du train
#     with open(os.path.join(data_root, 'train.txt')) as fd:
#         all_fnames = fd.read().strip().split('\n')

#     # --- 2. Filtrer uniquement les motions qui ont un fichier texte
#     text_dir = os.path.join(data_root, 'texts', 'texts')
#     available_texts = [f.replace('.txt','') for f in os.listdir(text_dir)]
#     all_fnames = [f for f in all_fnames if f in available_texts]

#     # --- 3. Split train / validation
#     random.shuffle(all_fnames)
#     split_idx = int((1 - val_ratio) * len(all_fnames))
#     train_fnames = all_fnames[:split_idx]
#     val_fnames = all_fnames[split_idx:]

#     print(f"Total motions: {len(all_fnames)}")
#     print(f"Train motions: {len(train_fnames)}")
#     print(f"Validation motions: {len(val_fnames)}")

#     # --- 4. Sauvegarder le nouveau fichier train_now.txt (sans les motions de validation)
#     train_now_path = os.path.join(save_dir, 'train_now.txt')
#     with open(train_now_path, 'w') as f:
#         for fname in train_fnames:
#             f.write(fname + '\n')
#     print(f"Saved new train split without validation motions to: {train_now_path}")

#     # --- 5. Sauvegarder aussi les IDs de validation pour référence
#     val_ids_path = os.path.join(save_dir, 'validation_ids.txt')
#     with open(val_ids_path, 'w') as f:
#         for fname in val_fnames:
#             f.write(fname + '\n')
#     print(f"Saved validation IDs to: {val_ids_path}")

#     # --- 6. Générer les batches de validation
#     if batch_size * num_batches > len(val_fnames):
#         raise ValueError("batch_size * num_batches > nombre de motions disponibles pour validation")

#     random_fnames = random.sample(val_fnames, k=batch_size * num_batches)
#     random_batches = np.array(random_fnames).reshape(num_batches, batch_size)

#     gt = [['query_id', 'candidate']]
#     mapping = []  # Pour garder trace des correspondances batch_id -> motion_id original
    
#     for file_idx, batch in enumerate(random_batches, start=1):
#         os.makedirs(pjoin(save_dir, str(file_idx)), exist_ok=True)

#         # --- Choisir la motion "correcte" pour le batch
#         rdm_idx = random.randint(0, len(batch)-1)
#         random_text_fname = batch[rdm_idx]

#         # --- Lire une description aléatoire pour la motion
#         with open(os.path.join(data_root, 'texts', 'texts', random_text_fname+'.txt')) as fd:
#             file_texts = fd.read().strip().split('\n')
#             random_text = random.choice(file_texts)

#         # --- Sauvegarder la query texte
#         with open(os.path.join(save_dir, str(file_idx), 'text.txt'), 'w') as fw:
#             fw.write(random_text)

#         # --- Copier toutes les motions du batch
#         for motion_idx, fname in enumerate(batch, start=1):
#             if fname == random_text_fname:
#                 gt.append([file_idx, motion_idx])
            
#             # Sauvegarder le mapping
#             mapping.append({
#                 'batch_id': file_idx,
#                 'position_in_batch': motion_idx,
#                 'original_motion_id': fname
#             })

#             motion = np.load(os.path.join(data_root, 'motions', 'motions', fname+'.npy'))
#             np.save(pjoin(save_dir, str(file_idx), f'motion_{motion_idx}.npy'), motion)

#     # --- Sauvegarder le fichier ground-truth
#     with open(os.path.join(save_dir, 'gt.csv'), 'w', newline='') as file:
#         writer = csv.writer(file)
#         writer.writerows(gt)

#     # --- Sauvegarder le mapping pour pouvoir retrouver les IDs originaux
#     mapping_df = pd.DataFrame(mapping)
#     mapping_df.to_csv(pjoin(save_dir, 'motion_id_mapping.csv'), index=False)
#     print(f"Saved motion ID mapping to: {pjoin(save_dir, 'motion_id_mapping.csv')}")

#     print(f'Done ! {num_batches} val batches saved at {save_dir}')

# # Exécution
# data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'
# save_dir = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val_split'

# batch_size = 32
# num_batches = 28
# val_ratio = 0.15  # 10% du train pour validation

# generate_val_batches_with_split(data_root, save_dir, batch_size, num_batches, val_ratio=val_ratio)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import os
from os.path import join as pjoin
import math
import random
import pandas as pd
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # évite deadlocks avec DataLoader multiprocess
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

# ==================== CONFIGURATION AMÉLIORÉE ====================
class ImprovedConfig:
    # Data
    data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'
    motion_dir = pjoin(data_root, 'motions', 'motions')
    text_dir = pjoin(data_root, 'texts', 'texts')
    train_split = pjoin(data_root, 'train.txt')
    val_dir = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val'
    model_checkpoint = './best_model_improved.pt'  # Chemin vers votre meilleur modèle
    # Motion processing
    motion_dim = 384
    d_model = 512  # Augmenté pour plus de capacité
    num_joints = 44
    pos_dim = 3
    rot_dim = 6
    
    # Text processing
    text_model_name = 'bert-base-uncased'
    text_dim = 768
    
    # Architecture améliorée (StrongMotionEncoder si use_strong_motion_encoder=True)
    num_heads = 16
    num_layers = 6
    use_strong_motion_encoder = True  # Conv 1D + 8 couches + proj plus profonde (pour ~90%)
    motion_conv_kernel = 5
    motion_strong_layers = 8
    dropout = 0.25
    max_motion_length = 200
    
    # Training (régularisation renforcée pour généraliser sur test Kaggle)
    batch_size = 32
    num_epochs = 50
    learning_rate = 1e-4
    weight_decay = 2e-3
    temperature = 0.01  
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Optimisation
    gradient_accumulation_steps = 2
    warmup_steps = 1000
    label_smoothing = 0.1
    
    # Augmentations (désactivées si use_augmentation=False)
    use_augmentation = False  # True = appliquer augmentation, False = aucune
    augmentation_prob = 0.3
    augmentation_for_retrieval = True  # si use_augmentation: True = mirror+bruit seul
    noise_std = 0.01
    mask_max_len = 20
    shift_max = 10


    # Mining
    hard_negative_k = 5
    mining_start_epoch = 5
    
    # Loss retrieval (ranking margin: positif doit être > meilleur négatif + margin)
    use_ranking_loss = True
    ranking_margin = 0.15  # m dans max(0, m - (sim_pos - sim_hard_neg))
    
    # Encoders
    use_mean_pool_text = True   # mean pool sur tokens (au lieu de CLS seul) pour le texte
    motion_norm_first = False   # False évite le warning Transformer et peut aider
    
    # Augmentation progressive (prob faible au début)
    augmentation_prob_start = 0.15
    augmentation_prob_max = 0.35
    augmentation_mirror_prob = 0.25   # flip gauche-droite squelette
    joint_dropout_prob = 0.1          # prob de mettre à 0 une dimension joint
    
    # Checkpoint (sauver top-K pour tester des epochs plus tôt sur Kaggle)
    save_path = './best_model.pt'
    chk_dir = './chk'
    save_top_k = 3
    patience = 20
    submission_file = 'submission.csv'
    
config = ImprovedConfig()
print(f"Using device: {config.device}")
print(f"Model dimension: {config.d_model}")
print(f"Batch size: {config.batch_size}")

# ==================== FONCTION D'ÉVALUATION ====================
def eval_recall(gt_df, pred_df, verbose=False):
    ks = list(range(1, 11))
    rank_cols = list(pred_df.columns)
    
    gt = gt_df["candidate"].values
    preds = pred_df[rank_cols].values
    
    assert len(gt) == len(preds), "size mismatch"
    
    recalls = {k: 0 for k in ks}
    n = len(gt)
    
    for i in range(n):
        for k in ks:
            if gt[i] in preds[i, :k]:
                recalls[k] += 1
    
    for k in ks:
        recalls[k] /= n
    
    weights = {k: 1.0 / k for k in ks}
    weight_sum = sum(weights.values())
    
    final_score = sum(weights[k] * recalls[k] for k in ks) / weight_sum
    
    if verbose:
        for k, val in recalls.items():
            print(f'recall@{k} = {round(val, 3)}')
    
    return final_score

def get_tokenizer(config):
    """Return tokenizer for current config (CLIP or BERT). Use for dataset and eval."""
    if getattr(config, 'use_clip_text', False):
        from transformers import CLIPProcessor
        proc = CLIPProcessor.from_pretrained(getattr(config, 'clip_model_name', 'openai/clip-vit-base-patch32'))
        return proc.tokenizer
    return AutoTokenizer.from_pretrained(config.text_model_name)

# ==================== MOTION ENCODER AMÉLIORÉ ====================
class ImprovedMotionEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Projection initiale avec normalisation
        self.input_proj = nn.Sequential(
            nn.Linear(config.motion_dim, config.d_model),
            nn.LayerNorm(config.d_model),
            nn.Dropout(config.dropout)
        )
        
        # Positional encoding apprenable
        self.pos_embedding = nn.Parameter(torch.randn(1, config.max_motion_length, config.d_model))
        
        # Transformer avec attention multi-têtes améliorée
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.num_heads,
            dim_feedforward=config.d_model * 4,
            dropout=config.dropout,
            activation='gelu',
            batch_first=True,
            norm_first=getattr(config, 'motion_norm_first', False)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)
        
        # Token [CLS] pour l'agrégation
        self.cls_token = nn.Parameter(torch.randn(1, 1, config.d_model))
        
        # Projection finale avec gating
        self.output_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model * 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_model * 2, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        
    def forward(self, motion, mask=None):
        batch_size = motion.shape[0]
        
        # Projection
        x = self.input_proj(motion)
        
        # Ajout position embedding
        seq_len = x.size(1)
        x = x + self.pos_embedding[:, :seq_len, :]
        
        # Ajout token [CLS]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        # Ajuster le mask
        if mask is not None:
            cls_mask = torch.ones(batch_size, 1, device=mask.device)
            extended_mask = torch.cat([cls_mask, mask], dim=1)
            transformer_mask = ~extended_mask.bool()
        else:
            transformer_mask = None
        
        # Transformer
        x = self.transformer(x, src_key_padding_mask=transformer_mask)
        
        # Utiliser le token CLS
        pooled = x[:, 0, :]
        
        # Projection finale
        embedding = self.output_proj(pooled)
        
        return F.normalize(embedding, dim=-1)

# ==================== MOTION ENCODER RENFORCÉ (pour ~90% recall) ====================
class StrongMotionEncoder(nn.Module):
    """Conv 1D (patterns locaux) + Transformer plus profond + projection plus riche."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        d_model = config.d_model
        k = getattr(config, 'motion_conv_kernel', 5)
        pad = k // 2
        self.conv = nn.Sequential(
            nn.Conv1d(config.motion_dim, d_model, kernel_size=k, padding=pad),
            nn.GELU(),
            nn.Dropout(config.dropout * 0.5),
        )
        self.conv_norm = nn.LayerNorm(d_model)
        self.linear_proj = nn.Linear(config.motion_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, config.max_motion_length, d_model) * 0.02)
        num_layers = getattr(config, 'motion_strong_layers', 8)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=config.num_heads,
            dim_feedforward=d_model * 4, dropout=config.dropout,
            activation='gelu', batch_first=True, norm_first=False
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model)
        )

    def forward(self, motion, mask=None):
        B, T, D = motion.shape
        x_conv = self.conv_norm(self.conv(motion.transpose(1, 2)).transpose(1, 2))
        x_lin = self.linear_proj(motion)
        x = (x_conv + x_lin) * 0.5 + self.pos_embedding[:, :T, :]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        if mask is not None:
            ext_mask = torch.cat([torch.ones(B, 1, device=mask.device), mask], dim=1)
            key_padding = ~ext_mask.bool()
        else:
            key_padding = None
        x = self.transformer(x, src_key_padding_mask=key_padding)
        pooled = x[:, 0, :]
        embedding = self.output_proj(pooled)
        return F.normalize(embedding, dim=-1)

# ==================== TEXT ENCODER ====================
class TextEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Load pre-trained BERT
        self.bert = AutoModel.from_pretrained(config.text_model_name)
        
        # Fine-tuning partiel de BERT
        # Geler les premières couches
        # for name, param in self.bert.named_parameters():
        #     if 'encoder.layer.0' in name or 'encoder.layer.1' in name:
        #         param.requires_grad = False
        
        # Projection
        self.output_proj = nn.Sequential(
            nn.Linear(config.text_dim, config.d_model * 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_model * 2, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # (B, L, D)
        if getattr(self.config, 'use_mean_pool_text', False):
            # Mean pooling over non-padding tokens (meilleur pour similarité sémantique)
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden[:, 0, :]
        embedding = self.output_proj(pooled)
        return F.normalize(embedding, dim=-1)

# ==================== MODÈLE PRINCIPAL AMÉLIORÉ ====================
class ImprovedTMRModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.motion_encoder = (StrongMotionEncoder(config) if getattr(config, 'use_strong_motion_encoder', False) else ImprovedMotionEncoder(config))
        self.text_encoder = TextEncoder(config)
        
        # Projections séparées
        self.text_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        self.motion_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        
        # Température apprenable
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / config.temperature))
        
    def forward(self, text_input_ids, text_attention_mask, motion, motion_mask=None):
        # Encodage
        text_emb = self.text_encoder(text_input_ids, text_attention_mask)
        motion_emb = self.motion_encoder(motion, motion_mask)
        
        # Projections spécifiques
        text_emb = self.text_proj(text_emb)
        motion_emb = self.motion_proj(motion_emb)
        
        # Normalisation
        text_emb = F.normalize(text_emb, dim=-1)
        motion_emb = F.normalize(motion_emb, dim=-1)
        
        return text_emb, motion_emb
    
    def _ranking_margin_loss(self, logits, scale):
        """Pousse le score du positif au-dessus du meilleur négatif d'au moins margin (meilleur recall)."""
        if not getattr(self.config, 'use_ranking_loss', False):
            return torch.tensor(0.0, device=logits.device)
        batch_size = logits.shape[0]
        margin = self.config.ranking_margin * scale  # en espace logit
        pos_score = logits.diag()
        # Pour chaque ligne, score max des négatifs (masquer la diagonale)
        logits_masked = logits.clone()
        logits_masked.fill_diagonal_(-1e9)
        hard_neg_score = logits_masked.max(dim=1).values
        loss_rank = F.relu(margin - (pos_score - hard_neg_score)).mean()
        return loss_rank

    def compute_loss(self, text_emb, motion_emb):
        scale = self.logit_scale.exp()
        logits = scale * torch.matmul(text_emb, motion_emb.T)
        
        batch_size = text_emb.shape[0]
        labels = torch.arange(batch_size, device=text_emb.device)
        
        loss_t2m = F.cross_entropy(logits, labels, label_smoothing=self.config.label_smoothing)
        loss_m2t = F.cross_entropy(logits.T, labels, label_smoothing=self.config.label_smoothing)
        loss_ce = (loss_t2m + loss_m2t) / 2
        
        loss_rank = self._ranking_margin_loss(logits, scale)
        loss = loss_ce + 0.5 * loss_rank
        
        with torch.no_grad():
            pred_t2m = logits.argmax(dim=-1)
            acc = (pred_t2m == labels).float().mean()
        
        return loss, acc
    
    def compute_loss_tmr(self, text_emb, motion_emb, neg_mask=None):
        """InfoNCE + ranking margin pour mieux corréler la loss avec le recall@1.
        neg_mask: (N,N) bool, True = filter out (wrong negative)."""
        tau = getattr(self.config, 'tmr_temperature', 0.1)
        tau = min(tau, getattr(self.config, 'tmr_tau_max', 0.5))  # plafonner pour éviter surconfiance
        S = torch.matmul(text_emb, motion_emb.T)  # cosine sim, (N, N)
        if neg_mask is not None:
            S = S.masked_fill(neg_mask, -1e4)  # -1e4 suffit pour softmax, évite overflow
        logits = (S / tau).clamp(-50.0, 50.0)  # clamp pour stabilité numérique
        N = logits.shape[0]
        labels = torch.arange(N, device=logits.device)
        ls = getattr(self.config, 'tmr_label_smoothing', 0.05)
        loss_t2m = F.cross_entropy(logits, labels, label_smoothing=ls)
        loss_m2t = F.cross_entropy(logits.T, labels, label_smoothing=ls)
        loss_nce = (loss_t2m + loss_m2t) / 2
        # Ranking margin: pousse le positif au-dessus du meilleur négatif (aligné recall@1)
        margin = getattr(self.config, 'tmr_ranking_margin', 0.2)
        pos_score = logits.diag()
        logits_masked = logits.clone()
        logits_masked.fill_diagonal_(-1e4)
        best_neg = logits_masked.max(dim=1).values
        loss_rank = F.relu(margin - (pos_score - best_neg)).mean()
        lambda_rank = getattr(self.config, 'tmr_lambda_rank', 0.5)
        loss = loss_nce + lambda_rank * loss_rank
        with torch.no_grad():
            acc = (logits.argmax(dim=-1) == labels).float().mean()
        return loss, acc
    
    def compute_loss_with_mining(self, text_emb, motion_emb, hard_negatives=None):
        """Loss avec hard negative mining: rehausser les logits des négatifs difficiles
        pour que la cross_entropy pénalise davantage quand le modèle les classe haut."""
        scale = self.logit_scale.exp()
        logits = scale * torch.matmul(text_emb, motion_emb.T)
        
        if hard_negatives is not None:
            # Copie pour ne pas modifier le tenseur utilisé pour acc
            logits = logits.clone()
            for i, neg_indices in enumerate(hard_negatives):
                for neg_idx in neg_indices.tolist():
                    if neg_idx != i:  # ne pas toucher au positif
                        logits[i, neg_idx] *= 1.2  # pénalité plus forte sur les hard neg
        
        batch_size = text_emb.shape[0]
        labels = torch.arange(batch_size, device=text_emb.device)
        
        loss_t2m = F.cross_entropy(logits, labels, label_smoothing=self.config.label_smoothing)
        loss_m2t = F.cross_entropy(logits.T, labels, label_smoothing=self.config.label_smoothing)
        loss_ce = (loss_t2m + loss_m2t) / 2
        loss_rank = self._ranking_margin_loss(logits, scale)
        loss = loss_ce + 0.5 * loss_rank
        
        with torch.no_grad():
            pred_t2m = (scale * torch.matmul(text_emb, motion_emb.T)).argmax(dim=-1)
            acc = (pred_t2m == labels).float().mean()
        
        return loss, acc

# ==================== AUGMENTATIONS ====================
class MotionAugmentation:
    def __init__(self, config):
        self.config = config
    
    def _prob(self):
        return getattr(self.config, 'augmentation_prob', 0.3)
    
    def time_mask(self, motion):
        if random.random() > self._prob():
            return motion
        motion = motion.clone()
        T = motion.shape[0]
        mask_len = random.randint(5, self.config.mask_max_len)
        start = random.randint(0, max(0, T - mask_len))
        motion[start:start+mask_len] = 0
        return motion
    
    def time_shift(self, motion):
        if random.random() > self._prob():
            return motion
        shift = random.randint(-self.config.shift_max, self.config.shift_max)
        device = motion.device
        if shift > 0:
            motion = torch.cat([torch.zeros(shift, motion.shape[1], device=device), motion[:-shift]])
        elif shift < 0:
            motion = torch.cat([motion[-shift:], torch.zeros(-shift, motion.shape[1], device=device)])
        return motion
    
    def gaussian_noise(self, motion):
        if random.random() > self._prob():
            return motion
        noise = torch.randn_like(motion, device=motion.device) * self.config.noise_std
        return motion + noise
    
    def speed_variation(self, motion):
        if random.random() > self._prob():
            return motion
        T = motion.shape[0]
        factor = random.uniform(0.8, 1.2)
        new_T = int(T * factor)
        device = motion.device
        indices = torch.linspace(0, T - 1, new_T, device=device)
        indices_floor = indices.long().clamp(0, T - 1)
        indices_ceil = (indices_floor + 1).clamp(max=T - 1)
        weights = (indices - indices_floor.float()).unsqueeze(-1)
        motion_aug = (1 - weights) * motion[indices_floor] + weights * motion[indices_ceil]
        if new_T < T:
            motion_aug = torch.cat([motion_aug, torch.zeros(T - new_T, motion.shape[1], device=device)])
        else:
            motion_aug = motion_aug[:T]
        return motion_aug
    
    def axis_mirror(self, motion):
        """Flip gauche-droite (négation X) pour les positions 3D des 2 personnes."""
        if random.random() > getattr(self.config, 'augmentation_mirror_prob', 0.25):
            return motion
        motion = motion.clone()
        # Layout: pos_p1 (0:66), rot_p1 (66:192), pos_p2 (192:258), rot_p2 (258:384). Négater X = indices 0,3,6,...,63 et 192,195,...,255
        for start, length in [(0, 66), (192, 66)]:
            for i in range(0, length, 3):
                motion[:, start + i] = -motion[:, start + i]
        return motion
    
    def joint_dropout(self, motion):
        """Met aléatoirement des dimensions (joints) à zéro."""
        if random.random() > getattr(self.config, 'joint_dropout_prob', 0.1):
            return motion
        motion = motion.clone()
        D = motion.shape[1]
        n_drop = max(1, int(D * 0.05))
        idx = torch.randperm(D, device=motion.device)[:n_drop]
        motion[:, idx] = 0
        return motion
    
    def __call__(self, motion):
        if not getattr(self.config, 'use_augmentation', True):
            return motion
        # Mode retrieval: seulement mirror + bruit léger
        if getattr(self.config, 'augmentation_for_retrieval', False):
            motion = self.axis_mirror(motion)
            motion = self.gaussian_noise(motion)
            return motion
        motion = self.time_mask(motion)
        motion = self.time_shift(motion)
        motion = self.speed_variation(motion)
        motion = self.axis_mirror(motion)
        motion = self.joint_dropout(motion)
        motion = self.gaussian_noise(motion)
        return motion

# ==================== HARD NEGATIVE MINING ====================
class HardNegativeMining:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.cache = {}
        
    @torch.no_grad()
    def get_hard_negatives(self, text_emb, motion_emb):
        """Trouve les exemples négatifs les plus difficiles"""
        # Matrice de similarité
        sim_matrix = torch.matmul(text_emb, motion_emb.T)
        
        batch_size = sim_matrix.shape[0]
        hard_negatives = []
        
        for i in range(batch_size):
            # Masquer la positive
            sim_matrix[i, i] = -float('inf')
            # Top-k négatifs
            _, indices = torch.topk(sim_matrix[i], self.config.hard_negative_k)
            hard_negatives.append(indices)
            
        return hard_negatives

# ==================== DATASET ====================
class MotionTextDataset(Dataset):
    """Dataset for text-motion pairs. When single_description_per_motion=True (default),
    each motion appears once per epoch with one of its descriptions; the description
    index rotates with epoch so the model sees all 3 descriptions across epochs.
    This keeps the contrastive diagonal correct (one positive per row)."""
    
    def __init__(self, config, split='train', single_description_per_motion=True):
        self.config = config
        self.single_description_per_motion = single_description_per_motion
        self.epoch = 0  # set via set_epoch() at start of each epoch
        
        # Load split
        with open(config.train_split, 'r') as f:
            all_motions = f.read().strip().split('\n')
        
        # Filter motions that have text files
        available_texts = set([f.replace('.txt', '') for f in os.listdir(config.text_dir)])
        self.motion_ids = [m for m in all_motions if m in available_texts]
        
        # Load tokenizer (CLIP or BERT via get_tokenizer)
        self.tokenizer = get_tokenizer(config)
        self.text_max_length = getattr(config, 'text_max_length', 77 if getattr(config, 'use_clip_text', False) else 64)
        
        # Pre-load all text descriptions
        self.texts = {}
        for motion_id in self.motion_ids:
            with open(pjoin(config.text_dir, f"{motion_id}.txt"), 'r') as f:
                self.texts[motion_id] = f.read().strip().split('\n')
        
        if single_description_per_motion:
            # One sample per motion; which description is chosen in __getitem__ via self.epoch
            self.samples = list(self.motion_ids)
            print(f"Dataset {split}: {len(self.samples)} motions, 1 description per motion per epoch (rotate by epoch)")
        else:
            # Original: all (motion_id, text_idx) pairs (diagonal no longer strict)
            self.samples = []
            for motion_id in self.motion_ids:
                for text_idx in range(len(self.texts[motion_id])):
                    self.samples.append((motion_id, text_idx))
            print(f"Dataset {split}: {len(self.samples)} samples from {len(self.motion_ids)} motions")
        
    def set_epoch(self, epoch):
        """Call at start of each epoch so each motion gets a deterministic but different description across epochs."""
        self.epoch = epoch
    
    def __len__(self):
        return len(self.samples)
    
    def _get_motion_id_and_text_idx(self, idx):
        if self.single_description_per_motion:
            motion_id = self.samples[idx]
            n_desc = len(self.texts[motion_id])
            text_idx = (self.epoch + idx) % n_desc
            return motion_id, text_idx
        else:
            return self.samples[idx][0], self.samples[idx][1]
    
    def __getitem__(self, idx):
        motion_id, text_idx = self._get_motion_id_and_text_idx(idx)
        
        # Load motion
        motion_path = pjoin(self.config.motion_dir, f"{motion_id}.npy")
        motion = np.load(motion_path)
        
        # Truncate or pad motion
        T = motion.shape[0]
        if T > self.config.max_motion_length:
            # Random crop for training
            start = np.random.randint(0, T - self.config.max_motion_length + 1)
            motion = motion[start:start + self.config.max_motion_length]
            mask = torch.ones(self.config.max_motion_length)
        else:
            # Pad
            padding = self.config.max_motion_length - T
            motion = np.pad(motion, ((0, padding), (0, 0)), mode='constant')
            mask = torch.cat([torch.ones(T), torch.zeros(padding)])
        
        # Load text
        text = self.texts[motion_id][text_idx]
        
        # Tokenize text
        encoded = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.text_max_length,
            return_tensors='pt'
        )
        
        return {
            'motion': torch.FloatTensor(motion),
            'motion_mask': mask,
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'motion_id': motion_id,
            'text': text  # for TMR negative filtering (text-text similarity)
        }

# ==================== TRAINING LOOP AMÉLIORÉE ====================
def train_epoch_improved(model, dataloader, optimizer, scheduler, config, epoch, miner=None):
    model.train()
    total_loss = 0
    total_acc = 0
    
    # Probabilité d'augmentation progressive (faible au début pour stabilité)
    p_start = getattr(config, 'augmentation_prob_start', 0.15)
    p_max = getattr(config, 'augmentation_prob_max', 0.35)
    config.augmentation_prob = min(p_max, p_start + (epoch / 10.0) * (p_max - p_start))
    
    augmenter = MotionAugmentation(config)
    
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Training")):
        motion = batch['motion']
        motion = torch.stack([augmenter(m) for m in motion])
        
        motion = motion.to(config.device)
        motion_mask = batch['motion_mask'].to(config.device)
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        
        # Forward
        text_emb, motion_emb = model(input_ids, attention_mask, motion, motion_mask)
        
        # Hard negative mining
        if miner is not None and epoch > config.mining_start_epoch:
            hard_negatives = miner.get_hard_negatives(text_emb, motion_emb)
            loss, acc = model.compute_loss_with_mining(text_emb, motion_emb, hard_negatives)
        else:
            loss, acc = model.compute_loss(text_emb, motion_emb)
        
        # Backward avec gradient accumulation
        loss = loss / config.gradient_accumulation_steps
        loss.backward()
        
        if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            # Scheduler par batch (sauf ReduceLROnPlateau, stepé par epoch sur le recall)
            if scheduler is not None and not isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step()
        
        total_loss += loss.item() * config.gradient_accumulation_steps
        total_acc += acc.item()
    
    return {
        'loss': total_loss / len(dataloader),
        'acc': total_acc / len(dataloader)
    }

# ==================== VALIDATION ====================
def evaluate_on_val_batches(model, val_dir, config, query_file='text.txt'):
    """Run retrieval on batch folders. Chaque batch contient query_file (ex: text.txt) et motion_1.npy .. motion_32.npy."""
    model.eval()
    
    batch_folders = sorted([f for f in os.listdir(val_dir) if f.isdigit()], key=int)
    predictions = []
    tokenizer = get_tokenizer(config)
    text_max_len = getattr(config, 'text_max_length', 77 if getattr(config, 'use_clip_text', False) else 64)
    
    with torch.no_grad():
        for batch_idx in batch_folders:
            batch_path = pjoin(val_dir, batch_idx)
            # Support both val (text.txt) and test (query.txt)
            query_path = pjoin(batch_path, query_file)
            if not os.path.isfile(query_path):
                query_path = pjoin(batch_path, 'query.txt' if query_file == 'text.txt' else 'text.txt')
            with open(query_path, 'r') as f:
                query_text = f.read().strip()
            
            # Tokenize query
            encoded = tokenizer(
                query_text,
                padding='max_length',
                truncation=True,
                max_length=text_max_len,
                return_tensors='pt'
            )
            input_ids = encoded['input_ids'].to(config.device)
            attention_mask = encoded['attention_mask'].to(config.device)
            
            # Get text embedding (CLIPRetrievalModel or ImprovedTMRModel)
            text_emb = model.text_encoder(input_ids, attention_mask)
            text_emb = model.text_proj(text_emb) if hasattr(model, 'text_proj') else text_emb
            text_emb = F.normalize(text_emb, dim=-1)
            
            # Load all motions in batch
            motion_scores = []
            motion_ids = []
            
            for i in range(1, 33):
                motion_path = pjoin(batch_path, f'motion_{i}.npy')
                motion = np.load(motion_path)
                
                # Process motion
                T = motion.shape[0]
                if T > config.max_motion_length:
                    start = (T - config.max_motion_length) // 2
                    motion = motion[start:start + config.max_motion_length]
                    mask = torch.ones(config.max_motion_length)
                else:
                    padding = config.max_motion_length - T
                    motion = np.pad(motion, ((0, padding), (0, 0)), mode='constant')
                    mask = torch.cat([torch.ones(T), torch.zeros(padding)])
                
                motion_tensor = torch.FloatTensor(motion).unsqueeze(0).to(config.device)
                mask_tensor = mask.unsqueeze(0).to(config.device)
                
                # Get motion embedding
                motion_emb = model.motion_encoder(motion_tensor, mask_tensor)
                motion_emb = model.motion_proj(motion_emb)
                motion_emb = F.normalize(motion_emb, dim=-1)
                
                # Compute similarity
                similarity = torch.matmul(text_emb, motion_emb.T).item()
                motion_scores.append(similarity)
                motion_ids.append(i)
            
            # Rank motions
            ranked_indices = np.argsort(motion_scores)[::-1][:10]
            ranked_motion_ids = [motion_ids[idx] for idx in ranked_indices]
            predictions.append([int(batch_idx)] + ranked_motion_ids)
    
    # Create submission DataFrame
    pred_df = pd.DataFrame(
        predictions,
        columns=['query_id'] + [f'candidate_{i}' for i in range(1, 11)]
    ).set_index('query_id')
    
    return pred_df


def evaluate_ensemble(checkpoint_paths, val_dir, config, query_file='text.txt'):
    """Prédictions en moyennant les scores de plusieurs modèles (meilleure généralisation test)."""
    models = []
    for path in checkpoint_paths:
        m = (CLIPRetrievalModel(config) if getattr(config, 'use_clip_text', False) else ImprovedTMRModel(config)).to(config.device)
        state = torch.load(path, map_location=config.device, weights_only=False)
        m.load_state_dict(state['model_state_dict'] if isinstance(state, dict) else state)
        m.eval()
        models.append(m)
    batch_folders = sorted([f for f in os.listdir(val_dir) if f.isdigit()], key=int)
    tokenizer = AutoTokenizer.from_pretrained(config.text_model_name)
    predictions = []
    with torch.no_grad():
        for batch_idx in batch_folders:
            batch_path = pjoin(val_dir, batch_idx)
            qpath = pjoin(batch_path, query_file)
            if not os.path.isfile(qpath):
                qpath = pjoin(batch_path, 'text.txt' if query_file != 'text.txt' else 'query.txt')
            with open(qpath, 'r') as f:
                query_text = f.read().strip()
            encoded = tokenizer(query_text, padding='max_length', truncation=True, max_length=64, return_tensors='pt')
            input_ids = encoded['input_ids'].to(config.device)
            attention_mask = encoded['attention_mask'].to(config.device)
            all_scores = []
            for m in models:
                text_emb = m.text_proj(m.text_encoder(input_ids, attention_mask))
                text_emb = F.normalize(text_emb, dim=-1)
                motion_scores = []
                for i in range(1, 33):
                    motion = np.load(pjoin(batch_path, f'motion_{i}.npy'))
                    T = motion.shape[0]
                    if T > config.max_motion_length:
                        start = (T - config.max_motion_length) // 2
                        motion = motion[start:start + config.max_motion_length]
                        mask = torch.ones(config.max_motion_length)
                    else:
                        padding = config.max_motion_length - T
                        motion = np.pad(motion, ((0, padding), (0, 0)), mode='constant')
                        mask = torch.cat([torch.ones(T), torch.zeros(padding)])
                    motion_t = torch.FloatTensor(motion).unsqueeze(0).to(config.device)
                    mask_t = mask.unsqueeze(0).to(config.device)
                    motion_emb = m.motion_proj(m.motion_encoder(motion_t, mask_t))
                    motion_emb = F.normalize(motion_emb, dim=-1)
                    motion_scores.append((text_emb @ motion_emb.T).item())
                all_scores.append(motion_scores)
            avg_scores = np.mean(all_scores, axis=0)
            ranked = np.argsort(avg_scores)[::-1][:10]
            predictions.append([int(batch_idx)] + [r+1 for r in ranked])
    pred_df = pd.DataFrame(predictions, columns=['query_id'] + [f'candidate_{i}' for i in range(1, 11)]).set_index('query_id')
    return pred_df


Using device: cuda
Model dimension: 512
Batch size: 32


## **Architecture TMR (Text-to-Motion Retrieval)**

Inspiré de [TMR (Petrovich et al., 2023)](https://arxiv.org/abs/2305.00976) :
- **InfoNCE** : perte contrastive (température τ=0.1) au lieu de cross-entropy + marge.
- **Filtrage des négatifs** : les paires (texte i, motion j) dont les textes i et j sont trop similaires (similarité > 0.8) sont exclues du calcul de la perte pour éviter de pousser des « faux négatifs ».
- Optionnel : installer `sentence-transformers` pour le filtrage : `pip install sentence-transformers`

Exécuter la cellule ci-dessous **avant** la boucle d'entraînement pour activer l'entraînement TMR.

In [13]:
# ==================== TMR: InfoNCE + Negative Filtering (arxiv.org/abs/2305.00976) ====================
# Override config for TMR (temperature 0.1, negative filtering threshold 0.8)
config.temperature = 0.1
config.tmr_temperature = 0.1
config.neg_filter_threshold = 0.8
config.use_negative_filtering = True
config.use_ranking_loss = False  # TMR uses only InfoNCE
config.label_smoothing = 0.0     # paper uses no label smoothing for InfoNCE
config.use_tmr_training = True   # use train_epoch_tmr in training loop
# Aligner loss et recall: ranking margin + label smoothing + plafond température
config.tmr_ranking_margin = 0.2   # m dans max(0, m - (sim_pos - sim_meilleur_neg))
config.tmr_lambda_rank = 0.5     # poids de la loss ranking
config.tmr_label_smoothing = 0.05
config.tmr_tau_max = 0.5         # plafond tau (évite surconfiance)
config.use_recall_scheduler = True   # ReduceLROnPlateau sur le recall val (meilleure corrélation)
config.recall_scheduler_patience = 3
config.recall_scheduler_factor = 0.5

# Optional: sentence-transformers for text-text similarity (filter "wrong negatives")
_tmr_text_encoder = None
try:
    from sentence_transformers import SentenceTransformer
    _tmr_text_encoder = SentenceTransformer('all-MiniLM-L6-v2')
    _tmr_text_encoder.eval()
    print("TMR: Using sentence_transformers for negative filtering.")
except Exception as e:
    print("TMR: sentence_transformers not available, negative filtering disabled.", e)

def get_negative_filter_mask(texts, threshold=0.8, device='cpu'):
    """Return (N, N) bool tensor: True = (i,j) is a wrong negative (texts i and j too similar), exclude from InfoNCE.
    Paper: filter pairs with text-text similarity above threshold (e.g. 0.8)."""
    if _tmr_text_encoder is None or not texts:
        return None
    import torch
    with torch.no_grad():
        emb = _tmr_text_encoder.encode(texts, convert_to_tensor=True, device=device)
        emb = emb / emb.norm(dim=1, keepdim=True)
        S = torch.mm(emb, emb.t())
    N = len(texts)
    mask = (S > threshold) & (~torch.eye(N, dtype=torch.bool, device=S.device))
    return mask

def train_epoch_tmr(model, dataloader, optimizer, scheduler, config, epoch, miner=None):
    """Training loop with TMR: InfoNCE loss + optional negative filtering (paper 2305.00976)."""
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    p_start = getattr(config, 'augmentation_prob_start', 0.15)
    p_max = getattr(config, 'augmentation_prob_max', 0.35)
    config.augmentation_prob = min(p_max, p_start + (epoch / 10.0) * (p_max - p_start))
    augmenter = MotionAugmentation(config)

    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Training TMR")):
        motion = torch.stack([augmenter(m) for m in batch['motion']])
        motion = motion.to(config.device)
        motion_mask = batch['motion_mask'].to(config.device)
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)

        text_emb, motion_emb = model(input_ids, attention_mask, motion, motion_mask)

        neg_mask = None
        if getattr(config, 'use_negative_filtering', False) and 'text' in batch:
            texts = batch['text']
            neg_mask = get_negative_filter_mask(texts, config.neg_filter_threshold, config.device)
            if neg_mask is not None:
                neg_mask = neg_mask.to(config.device)

        loss, acc = model.compute_loss_tmr(text_emb, motion_emb, neg_mask=neg_mask)

        loss = loss / getattr(config, 'gradient_accumulation_steps', 1)
        loss.backward()

        if (batch_idx + 1) % getattr(config, 'gradient_accumulation_steps', 1) == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            if scheduler is not None and not isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step()

        total_loss += loss.item() * getattr(config, 'gradient_accumulation_steps', 1)
        total_acc += acc.item()

    return {'loss': total_loss / len(dataloader), 'acc': total_acc / len(dataloader)}

2026-02-18 22:33:27.107266: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-18 22:33:27.774892: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-18 22:33:29.767513: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TMR: Using sentence_transformers for negative filtering.


## **Architecture CLIP-style (recommandée pour >75% score)**

Encodeur de **texte CLIP** (OpenAI/LAION) + **motion encoder** renforcé, perte contrastive symétrique (comme CLIP). Plus robuste que BERT seul pour l'alignement sémantique.

- **Texte** : CLIP ViT-B/32 (ou `laion/CLIP-ViT-B-32-laion2B-s34B-b79K`), option gelé ou fine-tuné.
- **Motion** : même encoder amélioré, projection vers 512-d.
- **Perte** : InfoNCE symétrique, température 0.07 (standard CLIP).
- **Batch** : 32–64, gradient accumulation si besoin.

Exécuter la cellule ci-dessous pour activer CLIP, puis **réexécuter la cellule du Dataset** (pour le tokenizer CLIP) et **créer le modèle avec `model = CLIPRetrievalModel(config)`** au lieu de `ImprovedTMRModel`.

In [14]:
# ==================== CLIP-style Text-to-Motion Retrieval ====================
# Activer CLIP : exécuter cette cellule puis recréer le dataset et le modèle avec CLIPRetrievalModel(config)

config.use_clip_text = True
config.clip_model_name = 'openai/clip-vit-base-patch32'  # ou 'laion/CLIP-ViT-B-32-laion2B-s34B-b79K' (plus fort)
config.text_max_length = 77   # max tokens CLIP
config.embed_dim = 512        # espace commun (sortie CLIP = 512)
config.d_model = 512          # motion encoder output
config.freeze_clip_text = True   # False + clip_text_lr_scale=0.05 pour fine-tune (viser 90%)
config.clip_text_lr_scale = 0.05 if not getattr(config, 'freeze_clip_text', True) else 0.1
config.clip_temperature = 0.07  # température CLIP standard
# Pour ~90%: augmentation_for_retrieval=True, use_strong_motion_encoder=True (déjà dans config)
# Réutiliser TMR si souhaité
config.use_tmr_training = getattr(config, 'use_tmr_training', False)

from transformers import CLIPTextModel, CLIPProcessor

class CLIPTextEncoder(nn.Module):
    """Encodeur de texte CLIP (OpenAI/LAION), sortie 512-d normalisée."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.clip_text = CLIPTextModel.from_pretrained(config.clip_model_name)
        self.proj_out = nn.Sequential(
            nn.Linear(self.clip_text.config.hidden_size, config.embed_dim),
            nn.LayerNorm(config.embed_dim),
        )
        if getattr(config, 'freeze_clip_text', True):
            for p in self.clip_text.parameters():
                p.requires_grad = False
            print("CLIP text encoder: frozen.")
        else:
            print("CLIP text encoder: trainable (use low LR).")

    def forward(self, input_ids, attention_mask):
        out = self.clip_text(input_ids=input_ids, attention_mask=attention_mask)
        # CLIP: utiliser pooler_output si dispo, sinon dernier token non-padding (EOS)
        if out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            last_hidden = out.last_hidden_state
            idx = (attention_mask.sum(dim=1) - 1).clamp(min=0)
            batch_size = last_hidden.shape[0]
            pooled = last_hidden[torch.arange(batch_size, device=last_hidden.device), idx]
        embedding = self.proj_out(pooled)
        return F.normalize(embedding, dim=-1)

class CLIPRetrievalModel(nn.Module):
    """Dual encoder CLIP-style: CLIP text + motion encoder, perte contrastive symétrique."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.text_encoder = CLIPTextEncoder(config)
        self.motion_encoder = (StrongMotionEncoder(config) if getattr(config, 'use_strong_motion_encoder', False) else ImprovedMotionEncoder(config))
        # Projections vers l'espace commun (motion encoder sort déjà en d_model=512)
        self.text_proj = nn.Sequential(
            nn.Linear(config.embed_dim, config.embed_dim),
            nn.LayerNorm(config.embed_dim),
        )
        self.motion_proj = nn.Sequential(
            nn.Linear(config.d_model, config.embed_dim),
            nn.LayerNorm(config.embed_dim),
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1.0 / getattr(config, 'clip_temperature', 0.07)))

    def forward(self, text_input_ids, text_attention_mask, motion, motion_mask=None):
        text_emb = self.text_encoder(text_input_ids, text_attention_mask)
        motion_emb = self.motion_encoder(motion, motion_mask)
        text_emb = self.text_proj(text_emb)
        motion_emb = self.motion_proj(motion_emb)
        text_emb = F.normalize(text_emb, dim=-1)
        motion_emb = F.normalize(motion_emb, dim=-1)
        return text_emb, motion_emb

    def compute_loss(self, text_emb, motion_emb):
        S = torch.matmul(text_emb, motion_emb.T).clamp(-1.0, 1.0)
        inv_tau = 1.0 / getattr(self.config, 'clip_temperature', 0.07)
        logits = (inv_tau * S).clamp(-50.0, 50.0)
        N = logits.shape[0]
        labels = torch.arange(N, device=logits.device)
        loss_t2m = F.cross_entropy(logits, labels)
        loss_m2t = F.cross_entropy(logits.T, labels)
        loss = (loss_t2m + loss_m2t) / 2
        with torch.no_grad():
            acc = (logits.argmax(dim=-1) == labels).float().mean()
        return loss, acc

    def compute_loss_tmr(self, text_emb, motion_emb, neg_mask=None):
        """TMR + ranking margin. Clamps pour éviter explosion (loss en millions)."""
        # Similarité cosinus: clamp au cas où embeddings ne seraient pas exactement normalisés
        S = torch.matmul(text_emb, motion_emb.T).clamp(-1.0, 1.0)
        if neg_mask is not None:
            S = S.masked_fill(neg_mask, -1e4)  # -1e4 suffit pour softmax, évite overflow
        # Scale fixe type CLIP (évite explosion du paramètre learnable)
        inv_tau = 1.0 / getattr(self.config, 'clip_temperature', 0.07)
        logits = (inv_tau * S).clamp(-50.0, 50.0)
        N = logits.shape[0]
        labels = torch.arange(N, device=logits.device)
        ls = getattr(self.config, 'tmr_label_smoothing', 0.05)
        loss_t2m = F.cross_entropy(logits, labels, label_smoothing=ls)
        loss_m2t = F.cross_entropy(logits.T, labels, label_smoothing=ls)
        loss_nce = (loss_t2m + loss_m2t) / 2
        margin = getattr(self.config, 'tmr_ranking_margin', 0.2)
        pos_score = logits.diag()
        logits_masked = logits.clone()
        logits_masked.fill_diagonal_(-1e4)
        best_neg = logits_masked.max(dim=1).values
        loss_rank = F.relu(margin - (pos_score - best_neg)).mean()
        lambda_rank = getattr(self.config, 'tmr_lambda_rank', 0.5)
        loss = loss_nce + lambda_rank * loss_rank
        with torch.no_grad():
            acc = (logits.argmax(dim=-1) == labels).float().mean()
        return loss, acc

# Créer le modèle CLIP (à utiliser à la place de ImprovedTMRModel)
# model = CLIPRetrievalModel(config).to(config.device)
# Si vous utilisez CLIP, recréez le dataset pour charger le tokenizer CLIP:
# dataset = MotionTextDataset(config);  train_loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, num_workers=0, pin_memory=True)

### **Recette pour viser ~90% recall**

- **Augmentation** : `augmentation_for_retrieval = True` (déjà dans config) — seulement mirror + bruit léger ; pas de time_mask / speed_variation / joint_dropout qui dégradent le sens du motion.
- **Motion encoder** : `use_strong_motion_encoder = True` (déjà dans config) — Conv 1D + 8 couches Transformer + projection plus profonde.
- **Optionnel** : dégeler le texte CLIP (`freeze_clip_text = False`) et utiliser `clip_text_lr_scale = 0.05` pour adapter le texte au domaine “motion”.
- **Optionnel** : augmenter `batch_size` à 64 si la GPU le permet (plus de négatifs par batch).
- **Optionnel** : checkpoint LAION pour le texte : `clip_model_name = 'laion/CLIP-ViT-B-32-laion2B-s34B-b79K'`.

In [15]:
# ==================== MAIN TRAINING SCRIPT AMÉLIORÉ ====================
def main():
    # Dataset: une description par motion par epoch (diagonale stricte pour le contraste
    dataset = MotionTextDataset(config, split='train', single_description_per_motion=True)
    
    # Pas de random_split: on utilise tout le train; la val est sur val_split/ (batches séparés)
    train_loader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True
    )
    
    # Create model
    model = (CLIPRetrievalModel(config) if getattr(config, 'use_clip_text', False) else ImprovedTMRModel(config)).to(config.device)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Optimizer avec param groups
    lr_text = config.learning_rate * getattr(config, 'clip_text_lr_scale', 0.1)  # 0.05 si CLIP unfreeze
    optimizer = torch.optim.AdamW([
        {'params': model.motion_encoder.parameters(), 'lr': config.learning_rate},
        {'params': model.text_encoder.parameters(), 'lr': lr_text},
        {'params': model.text_proj.parameters(), 'lr': config.learning_rate},
        {'params': model.motion_proj.parameters(), 'lr': config.learning_rate},
        {'params': model.logit_scale, 'lr': config.learning_rate}
    ], weight_decay=config.weight_decay)
    
    # Scheduler: ReduceLROnPlateau sur recall val (meilleure corrélation loss/recall) ou cosine
    if getattr(config, 'use_recall_scheduler', False):
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=getattr(config, 'recall_scheduler_factor', 0.5),
            patience=getattr(config, 'recall_scheduler_patience', 3), min_lr=1e-6
        )
    else:
        num_training_steps = (len(train_loader) // config.gradient_accumulation_steps) * config.num_epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=config.warmup_steps, num_training_steps=max(num_training_steps, 1)
        )
    
    # Hard negative miner
    miner = HardNegativeMining(model, config)
    
    # Training loop
    best_recall = 0
    patience = getattr(config, 'patience', 6)
    patience_counter = 0
    top_k_checkpoints = []  # list of (recall, epoch, path)
    os.makedirs(getattr(config, 'chk_dir', './chk'), exist_ok=True)
    
    for epoch in range(config.num_epochs):
        # Changer la description utilisée pour chaque motion (rotation parmi les 3)
        dataset.set_epoch(epoch)
        
        print(f"\nEpoch {epoch + 1}/{config.num_epochs}")
        print(f"Learning rate: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Train (TMR if config.use_tmr_training else baseline)
        train_fn = train_epoch_tmr if getattr(config, 'use_tmr_training', False) else train_epoch_improved
        train_metrics = train_fn(model, train_loader, optimizer, scheduler, config, epoch, miner)
        print(f"Train - Loss: {train_metrics['loss']:.4f}, Acc: {train_metrics['acc']:.4f}")
        
        # Validate
        pred_df = evaluate_on_val_batches(model, config.val_dir, config)
        gt_df = pd.read_csv(pjoin(config.val_dir, 'gt.csv'))
        recall_score = eval_recall(gt_df, pred_df, verbose=True)
        print(f"Validation Recall Score: {recall_score:.4f}")
        
        # Step scheduler sur le recall (ReduceLROnPlateau)
        if getattr(config, 'use_recall_scheduler', False):
            scheduler.step(recall_score)
        
        # Save best model
        if recall_score > best_recall:
            best_recall = recall_score
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_recall': best_recall,
            }, config.save_path)
            chk_dir = getattr(config, 'chk_dir', './chk')
            ep_path = pjoin(chk_dir, f'best_epoch_{epoch+1}.pt')
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'best_recall': best_recall}, ep_path)
            top_k_checkpoints.append((recall_score, epoch+1, ep_path))
            top_k_checkpoints.sort(key=lambda x: -x[0])
            save_top_k = getattr(config, 'save_top_k', 5)
            while len(top_k_checkpoints) > save_top_k:
                _, _, old_path = top_k_checkpoints.pop()
                if os.path.isfile(old_path):
                    os.remove(old_path)
            print(f"New best model saved with recall: {best_recall:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
            
        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping after {epoch + 1} epochs")
            break
    
    print(f"\nTraining completed. Best Recall (val): {best_recall:.4f}")
    if top_k_checkpoints:
        print("Top checkpoints (si le score Kaggle baisse, essayer un epoch plus tôt ou l'ensemble):")
        for r, ep, path in sorted(top_k_checkpoints, key=lambda x: -x[0]):
            print(f"  epoch {ep}  recall val={r:.4f}  -> {path}")
    
    # Load best model for final evaluation
    checkpoint = torch.load(config.save_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Final evaluation
    pred_df = evaluate_on_val_batches(model, config.val_dir, config)
    gt_df = pd.read_csv(pjoin(config.val_dir, 'gt.csv'))
    final_score = eval_recall(gt_df, pred_df, verbose=True)
    print(f"\nFinal Recall Score with best model: {final_score:.4f}")

if __name__ == "__main__":
    main()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Dataset train: 6018 motions, 1 description per motion per epoch (rotate by epoch)
CLIP text encoder: frozen.
Total parameters: 92,561,409
Trainable parameters: 29,395,457

Epoch 1/50
Learning rate: 1.00e-04


Training TMR: 100%|██████████| 188/188 [01:10<00:00,  2.69it/s]


Train - Loss: 3.6144, Acc: 0.0532


/home/layesekou/masterds/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


recall@1 = 0.133
recall@2 = 0.233
recall@3 = 0.233
recall@4 = 0.267
recall@5 = 0.367
recall@6 = 0.4
recall@7 = 0.467
recall@8 = 0.567
recall@9 = 0.567
recall@10 = 0.633
Validation Recall Score: 0.2725
New best model saved with recall: 0.2725

Epoch 2/50
Learning rate: 1.00e-04


... (22 outputs left out)

Training TMR: 100%|██████████| 188/188 [01:07<00:00,  2.79it/s]


Train - Loss: 2.5000, Acc: 0.3725
recall@1 = 0.4
recall@2 = 0.6
recall@3 = 0.667
recall@4 = 0.7
recall@5 = 0.767
recall@6 = 0.867
recall@7 = 0.9
recall@8 = 0.9
recall@9 = 0.9
recall@10 = 0.933
Validation Recall Score: 0.6246

Epoch 14/50
Learning rate: 1.00e-04


Training TMR:  20%|██        | 38/188 [00:15<00:59,  2.50it/s]


KeyboardInterrupt: 

In [20]:
# Load best model, run inference on test set and save submission CSV
test_dir = pjoin(data_root, 'test', 'test')
model = (CLIPRetrievalModel(config) if getattr(config, 'use_clip_text', False) else ImprovedTMRModel(config)).to(config.device)
config.save_path = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/chk/best_epoch_14.pt' 

# Load checkpoint with weights_only=False to handle custom Config class
state = torch.load(config.save_path, map_location=config.device, weights_only=False)
if isinstance(state, dict) and 'model_state_dict' in state:
    model.load_state_dict(state['model_state_dict'])
else:
    model.load_state_dict(state)

model.eval()
# Option 1: un seul modèle (best)
pred_df = evaluate_on_val_batches(model, test_dir, config)
# Option 2 (si score Kaggle < val): ensemble de plusieurs checkpoints pour mieux généraliser
# chk_paths = [pjoin(config.chk_dir, f) for f in ['best_epoch_28.pt', 'best_epoch_30.pt', 'best_epoch_38.pt'] if os.path.exists(pjoin(config.chk_dir, f))]
# if chk_paths:
#     pred_df = evaluate_ensemble(chk_paths, test_dir, config)

# Ensure predictions are in the correct format for submission
submission_df = pred_df.reset_index()
submission_df.columns = ['query_id'] + [f'candidate_{i}' for i in range(1, 11)]

# Verify the format
print("Submission format:")
print(submission_df.head())
print(f"\nShape: {submission_df.shape}")
print(f"Columns: {submission_df.columns.tolist()}")

# Save to CSV
submission_path = pjoin(data_root, 'submission.csv')
submission_df.to_csv(submission_path, index=False)
print(f"\nSubmission saved to: {submission_path}")

# Verify saved file
verify_df = pd.read_csv(submission_path)
print("\nVerification - First 3 rows of saved submission:")
print(verify_df.head(3))


CLIP text encoder: frozen.
Submission format:
   query_id  candidate_1  candidate_2  candidate_3  candidate_4  candidate_5  \
0         1           17           15           30            9           28   
1         2           18           21           10           16           11   
2         3            7           16           30           22           18   
3         4           20            2            7            1           24   
4         5           17            6            3           20           18   

   candidate_6  candidate_7  candidate_8  candidate_9  candidate_10  
0            6            8           23           32             5  
1            7           24            3            9            14  
2           28           21            6            1             3  
3           31           28           32           15            22  
4            8           10           24           28             7  

Shape: (30, 11)
Columns: ['query_id', 'candidate_1',

In [ ]:
# Load checkpoint with weights_only=False to handle custom Config class
state = torch.load(config.save_path, map_location=config.device, weights_only=False)
if isinstance(state, dict) and 'model_state_dict' in state:
    model.load_state_dict(state['model_state_dict'])
else:
    model.load_state_dict(state)